In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, silhouette_score
from sklearn.decomposition import PCA
from scipy.stats import chi2
from sklearn.mixture import GaussianMixture

In [2]:
def matriz_conf(y_true, y_pred, labels):
    total_labels = labels
    print(total_labels)
    cm = np.zeros((len(total_labels),len(total_labels)), dtype=int)
    for i in range(len(y_true)):
        cm[y_true[i]][y_pred[i]] += 1

    
    cm = pd.DataFrame(cm, columns=total_labels, index=total_labels)

    cm_transp = pd.DataFrame(np.transpose(cm.to_numpy()), columns=total_labels, index=total_labels)

    for c in cm_transp.columns:
        cm_transp[c] = cm_transp[c]/cm_transp[c].sum()

    cm_porcento = pd.DataFrame(np.transpose(cm_transp.to_numpy()), columns=total_labels, index=total_labels)

    return cm, cm_porcento

def acc(cm, hidden_classes):
    cm_transp = pd.DataFrame(np.transpose(cm.dropna().to_numpy()), columns=cm.columns, index=cm.columns)
    acc = 0
    total = 0
    for c in cm_transp.columns:
        if c not in hidden_classes:
            acc += cm_transp[c][c]
        else:
            acc += cm_transp[c][-1]
        total += cm_transp[c].sum()
    return acc/total

In [3]:
filenames = [0,2,3,4,5]

labels_str = ['DDoS', 'Benign', 'DoS', 'BruteForce', 'Bot', 'Web']

filenames

# pd.set_option('future.no_silent_downcasting', True)

[0, 2, 3, 4, 5]

In [4]:
exp_val = []
y_true_all_exp_val = []
for i in range(len(filenames)):
    exp_val.append(pd.read_csv(f'val_{filenames[i]}_GMM_BIC_1_10_scores_corr.csv'))
    y_true_all_exp_val.append(exp_val[i]['Label'].values.tolist())
    exp_val[i] = exp_val[i].drop(columns=['Label'])

In [7]:
from sklearn.metrics import classification_report
ths = [20, 21, 22, 23, 24, 25, 26, 27]
f1s = [[],[],[],[],[],[],[],[]]
for i in range(len(exp_val)):
    index = 0
    for th in ths:
        y_pred = []
        prog= 0
        for j in range(len(exp_val[i])):
            # print(exp[i][j])
            m = np.nanmax(exp_val[i].values[j])
            # print(m)
            if m > th:
                y_pred.append(exp_val[i].values[j].tolist().index(m))
            else:
                y_pred.append(-1)
        # print(y_pred[:10])

        cm, cm_porcento = matriz_conf(y_true_all_exp_val[i], y_pred, list(range(len(labels_str))) + [-1])
        print(f'th = {th} hidden = {filenames[i]}')
        display(cm)
        tp = cm[-1][filenames[i]]
        fp = cm[-1].sum() - tp
        fn = cm.iloc[filenames[i]].sum() - tp
        tn = cm.drop(columns=[-1]).values.sum() - fn

        acc = (tp+tn)/(tp+fp+tn+fn)
        recall = tp/(tp+fn)
        precision = tp/(tp+fp)
        if precision == 0 or recall == 0:
            f1 = 0
        else:
            f1 = 2*precision*recall/(precision+recall)

        f1s[index].append(f1)
        index += 1
        print(f'th: {th} hidden: {filenames[i]} acc:{acc} recall:{recall} precision:{precision} f1:{f1}')

        true_labels = np.array(y_true_all_exp_val[i])
        true_labels[true_labels == filenames[i]] = -1

        print('MULTICLASS DETECTION')
        print(classification_report(true_labels, y_pred))

[0, 1, 2, 3, 4, 5, -1]
th = 20 hidden = 0


,0,1,2,3,4,5,-1
0,0,2272,0,0,0,0,199956
1,0,123518,13,1,248,145,4611
2,0,1,81919,0,0,0,385
3,0,0,3,60934,0,0,13
4,0,8,0,0,45645,41,96
5,0,14,0,0,8,102,23
-1,0,0,0,0,0,0,0


th: 20 hidden: 0 acc:0.9857680265253214 recall:0.9887651561603734 precision:0.9749956115542899 f1:0.9818321090466277
MULTICLASS DETECTION
              precision    recall  f1-score   support

          -1       0.97      0.99      0.98    202228
           1       0.98      0.96      0.97    128536
           2       1.00      1.00      1.00     82305
           3       1.00      1.00      1.00     60950
           4       0.99      1.00      1.00     45790
           5       0.35      0.69      0.47       147

    accuracy                           0.98    519956
   macro avg       0.88      0.94      0.90    519956
weighted avg       0.99      0.98      0.98    519956

[0, 1, 2, 3, 4, 5, -1]
th = 21 hidden = 0


,0,1,2,3,4,5,-1
0,0,2259,0,0,0,0,199969
1,0,122335,13,1,195,124,5868
2,0,0,81827,0,0,0,478
3,0,0,3,60934,0,0,13
4,0,8,0,0,45586,0,196
5,0,14,0,0,8,88,37
-1,0,0,0,0,0,0,0


th: 21 hidden: 0 acc:0.9829774057804891 recall:0.9888294400379769 precision:0.968086908951835 f1:0.9783482432257228
MULTICLASS DETECTION
              precision    recall  f1-score   support

          -1       0.97      0.99      0.98    202228
           1       0.98      0.95      0.97    128536
           2       1.00      0.99      1.00     82305
           3       1.00      1.00      1.00     60950
           4       1.00      1.00      1.00     45790
           5       0.42      0.60      0.49       147

    accuracy                           0.98    519956
   macro avg       0.89      0.92      0.90    519956
weighted avg       0.98      0.98      0.98    519956

[0, 1, 2, 3, 4, 5, -1]
th = 22 hidden = 0


...

th: 25 hidden: 5 acc:0.9299863834632162 recall:0.6462585034013606 precision:0.00260652454248635 f1:0.005192107995846313
MULTICLASS DETECTION
              precision    recall  f1-score   support

          -1       0.00      0.65      0.01       147
           0       1.00      0.98      0.99    202228
           1       1.00      0.80      0.89    128536
           2       1.00      0.95      0.98     82305
           3       1.00      1.00      1.00     60950
           4       1.00      0.95      0.97     45790

    accuracy                           0.93    519956
   macro avg       0.83      0.89      0.81    519956
weighted avg       1.00      0.93      0.96    519956

[0, 1, 2, 3, 4, 5, -1]
th = 26 hidden = 5


,0,1,2,3,4,5,-1
0,193225,0,0,0,0,0,9003
1,0,94283,3,0,0,0,34250
2,0,0,74820,0,0,0,7485
3,0,0,3,60892,0,0,55
4,0,0,0,0,41471,0,4319
5,0,18,0,0,6,0,123
-1,0,0,0,0,0,0,0


th: 26 hidden: 5 acc:0.8939602581756918 recall:0.8367346938775511 precision:0.002226848918258351 f1:0.004441876421942148
MULTICLASS DETECTION
              precision    recall  f1-score   support

          -1       0.00      0.84      0.00       147
           0       1.00      0.96      0.98    202228
           1       1.00      0.73      0.85    128536
           2       1.00      0.91      0.95     82305
           3       1.00      1.00      1.00     60950
           4       1.00      0.91      0.95     45790

    accuracy                           0.89    519956
   macro avg       0.83      0.89      0.79    519956
weighted avg       1.00      0.89      0.94    519956

[0, 1, 2, 3, 4, 5, -1]
th = 27 hidden = 5


,0,1,2,3,4,5,-1
0,189367,0,0,0,0,0,12861
1,0,75150,3,0,0,0,53383
2,0,0,71083,0,0,0,11222
3,0,0,3,60808,0,0,139
4,0,0,0,0,40125,0,5665
5,0,0,0,0,3,0,144
-1,0,0,0,0,0,0,0


th: 27 hidden: 5 acc:0.8398460638977144 recall:0.9795918367346939 precision:0.001726328913611624 f1:0.003446583932695875
MULTICLASS DETECTION
              precision    recall  f1-score   support

          -1       0.00      0.98      0.00       147
           0       1.00      0.94      0.97    202228
           1       1.00      0.58      0.74    128536
           2       1.00      0.86      0.93     82305
           3       1.00      1.00      1.00     60950
           4       1.00      0.88      0.93     45790

    accuracy                           0.84    519956
   macro avg       0.83      0.87      0.76    519956
weighted avg       1.00      0.84      0.90    519956



# Média absolute threshold

In [8]:
for i in range(len(f1s)):
    print(f'Média f1 absolute th {ths[i]}: {np.mean(np.array(f1s[i]))}')

Média f1 absolute th 20: 0.5007415754543887
Média f1 absolute th 21: 0.6584299012373065
Média f1 absolute th 22: 0.6571573122750525
Média f1 absolute th 23: 0.650093386492642
Média f1 absolute th 24: 0.6235482825706782
Média f1 absolute th 25: 0.5991093246813098
Média f1 absolute th 26: 0.5572627356056554
Média f1 absolute th 27: 0.5599554477676085
